# Spam Detection Notebook (Beginner Step-by-Step)

This notebook is for first-time learners.

How to use it:
- Run one cell at a time from top to bottom.
- Read output before moving to next cell.
- If error appears, fix that step and rerun.

Goal: predict whether an email is `spam` or `ham`.

## Step 0: Install and Import Libraries

What is happening here:
- We install and import all tools we need.
- We also set random seeds so your result is more repeatable.

Why this step matters:
- If libraries are missing, the notebook cannot run.
- Reproducibility is important in machine learning learning and debugging.

In [ ]:
# Cell 3: Optional install command if packages are missing.
# Run this only once if imports fail in the next cell.
# %pip install numpy pandas scikit-learn matplotlib seaborn nltk wordcloud tensorflow

In [ ]:
# Cell 4: Import all libraries and set random seeds for stable results.
from pathlib import Path
import string
import random

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

from nltk import download as nltk_download
from nltk.corpus import stopwords
from wordcloud import WordCloud

nltk_download('stopwords', quiet=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('All libraries loaded.')

## Step 1: Locate and Load the CSV

What is happening here:
- We search common folders for `spam_ham_dataset.csv`.
- We load it into a pandas DataFrame.
- We validate required columns: `label` and `text`.

Why this step matters:
- Clean input format avoids hidden bugs later.
- If these columns are missing, model training will fail.

In [ ]:
# Cell 6: Find CSV file, load it, and validate required columns.
def locate_csv() -> Path:
    candidates = [
        Path.cwd() / 'spam_ham_dataset.csv',
        Path.cwd() / 'data' / 'raw' / 'spam_ham_dataset.csv',
        Path.cwd().parent / 'spam_ham_dataset.csv',
        Path.cwd().parent / 'data' / 'raw' / 'spam_ham_dataset.csv',
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError('Could not find spam_ham_dataset.csv in expected paths.')

csv_path = locate_csv()
data = pd.read_csv(csv_path)

required = {'label', 'text'}
missing = required.difference(data.columns)
if missing:
    raise ValueError(f'Missing required columns: {sorted(missing)}')

print(f'Loaded: {csv_path}')
print(f'Shape: {data.shape}')
data.head()

## Step 2: Understand Class Distribution

What is happening here:
- We count how many messages are `ham` and how many are `spam`.

Why this step matters:
- If one class is much larger, the model can become biased.
- We inspect this first so we can decide whether balancing is needed.

In [ ]:
# Cell 8: Plot class counts to check data balance (ham vs spam).
plt.figure(figsize=(7, 4))
sns.countplot(x='label', data=data)
plt.title('Original Class Distribution')
plt.tight_layout()
plt.show()

## Step 3: Balance the Dataset

What is happening here:
- We keep all `spam` rows.
- We randomly sample the same number of `ham` rows.
- Then we shuffle the rows.

Why this step matters:
- Balanced classes make training fairer.
- Accuracy becomes more meaningful for both classes.

In [ ]:
# Cell 10: Balance classes by downsampling ham to match spam size.
ham_df = data[data['label'].str.lower() == 'ham']
spam_df = data[data['label'].str.lower() == 'spam']

if ham_df.empty or spam_df.empty:
    raise ValueError('Both ham and spam classes are required.')

ham_balanced = ham_df.sample(n=len(spam_df), random_state=SEED)
balanced_data = pd.concat([ham_balanced, spam_df], axis=0)
balanced_data = balanced_data.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

print('Balanced shape:', balanced_data.shape)
plt.figure(figsize=(7, 4))
sns.countplot(x='label', data=balanced_data)
plt.title('Balanced Class Distribution')
plt.tight_layout()
plt.show()

## Step 4: Clean the Text

What is happening here:
- Convert text to lowercase.
- Remove the frequent email token `subject`.
- Remove punctuation.
- Remove common stopwords like `the`, `is`, `and`.

Why this step matters:
- Reduces noise in text.
- Helps the model focus on words that carry useful meaning.

In [ ]:
# Cell 12: Clean text by lowercasing, removing punctuation, and stopwords.
def remove_punctuation(text: str) -> str:
    table = str.maketrans('', '', string.punctuation)
    return str(text).translate(table)


def remove_stop_words(text: str) -> str:
    sw = set(stopwords.words('english'))
    words = str(text).lower().split()
    words = [w for w in words if w not in sw]
    return ' '.join(words)

cleaned_data = balanced_data.copy()
cleaned_data['text'] = cleaned_data['text'].astype(str).str.lower()
cleaned_data['text'] = cleaned_data['text'].str.replace('subject', '', regex=False)
cleaned_data['text'] = cleaned_data['text'].apply(remove_punctuation)
cleaned_data['text'] = cleaned_data['text'].apply(remove_stop_words)

cleaned_data[['label', 'text']].head()

## Step 5: Visualize Word Clouds

What is happening here:
- We create one word cloud for `ham` and one for `spam`.

Why this step matters:
- This gives a human-understandable view of what words dominate each class.
- It is a quick sanity check before modeling.

In [ ]:
# Cell 14: Show word clouds to visualize common words in each class.
def plot_wordcloud(df: pd.DataFrame, label_name: str, title: str) -> None:
    text_blob = ' '.join(df[df['label'].str.lower() == label_name]['text'].astype(str).tolist())
    if not text_blob.strip():
        print(f'No text for label: {label_name}')
        return

    wc = WordCloud(background_color='black', width=900, height=450, max_words=120).generate(text_blob)
    plt.figure(figsize=(10, 5))
    plt.imshow(wc, interpolation='bilinear')
    plt.title(title)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

plot_wordcloud(cleaned_data, 'ham', 'WordCloud: Ham')
plot_wordcloud(cleaned_data, 'spam', 'WordCloud: Spam')

## Step 6: Tokenization and Padding

What is happening here:
- Tokenization maps each word to an integer id.
- Padding makes every message the same length.
- We split train and test sets with stratification.

Why this step matters:
- Neural networks cannot read raw text directly.
- Equal-length sequences are required for batch training.

In [ ]:
# Cell 16: Convert text to numbers (tokenize) and equal lengths (pad).
labels = (cleaned_data['label'].str.lower() == 'spam').astype(int).to_numpy()

X_train_text, X_test_text, y_train, y_test = train_test_split(
    cleaned_data['text'],
    labels,
    test_size=0.2,
    random_state=SEED,
    stratify=labels,
)

MAX_WORDS = 12000
MAX_LEN = 100

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train_text.tolist())

X_train_seq = tokenizer.texts_to_sequences(X_train_text.tolist())
X_test_seq = tokenizer.texts_to_sequences(X_test_text.tolist())

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding='post', truncating='post')

print('Train tensor shape:', X_train_pad.shape)
print('Test tensor shape :', X_test_pad.shape)

## Step 7: Build the Model (Why LSTM?)

What this cell does:
- Creates Embedding + LSTM + Dense model.
- Compiles model with Adam and binary cross-entropy.

Why we chose LSTM:
- Email text is a sequence, and word order matters.
- LSTM captures context better than simple word-count methods.
- It is easier for beginners than large Transformer models.

Later you can compare with: TF-IDF + Logistic Regression, GRU, and BERT.

In [ ]:
# Cell 18: Build a simple LSTM model for spam/ham prediction.
vocab_size = min(MAX_WORDS, len(tokenizer.word_index) + 1)

model = tf.keras.Sequential([
    tf.keras.layers.Embedding(input_dim=vocab_size, output_dim=32, input_length=MAX_LEN),
    tf.keras.layers.LSTM(16),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid'),
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy'],
)

model.summary()

## Step 8: Train the Model

What is happening here:
- We train for multiple epochs.
- `EarlyStopping` stops training when validation stops improving.
- `ReduceLROnPlateau` lowers learning rate when progress slows.

Why this step matters:
- Prevents overfitting.
- Improves stability and often improves final validation performance.

In [ ]:
# Cell 20: Train the model with callbacks to avoid overfitting.
early_stop = EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1)

history = model.fit(
    X_train_pad,
    y_train,
    validation_data=(X_test_pad, y_test),
    epochs=20,
    batch_size=32,
    callbacks=[early_stop, reduce_lr],
    verbose=1,
)

## Step 9: Evaluate Performance

What is happening here:
- We evaluate on test data that was not used for training.

Why this step matters:
- This is the best quick estimate of real-world performance.
- Training accuracy alone can be misleading.

In [ ]:
# Cell 22: Evaluate model with accuracy, report, and confusion matrix.
test_loss, test_acc = model.evaluate(X_test_pad, y_test, verbose=0)
print(f'Test Loss: {test_loss:.4f}')
print(f'Test Accuracy: {test_acc:.4f}')

probs = model.predict(X_test_pad, verbose=0).ravel()
preds = (probs >= 0.5).astype(int)

print('\nClassification Report:')
print(classification_report(y_test, preds, target_names=['ham', 'spam']))

cm = confusion_matrix(y_test, preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

## Step 10: Plot Accuracy Curve

What is happening here:
- We plot training and validation accuracy per epoch.

Why this step matters:
- Helps visualize underfitting and overfitting.
- If curves diverge strongly, we may need regularization or data changes.

In [ ]:
# Cell 24: Plot learning curves (accuracy and loss).
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history.get('accuracy', []), label='Train Accuracy')
axes[0].plot(history.history.get('val_accuracy', []), label='Val Accuracy')
axes[0].set_title('Accuracy by Epoch')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].plot(history.history.get('loss', []), label='Train Loss')
axes[1].plot(history.history.get('val_loss', []), label='Val Loss')
axes[1].set_title('Loss by Epoch')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.show()

## Step 11: Save Model and Report

What is happening here:
- Save the trained model to disk.
- Save test metrics to a text report.

Why this step matters:
- You can load the model later without retraining.
- Keeping metrics helps compare future experiments.

In [ ]:
# Cell 26: Save trained model and key metrics to files.
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
artifacts = project_root / 'artifacts'
(artifacts / 'models').mkdir(parents=True, exist_ok=True)
(artifacts / 'reports').mkdir(parents=True, exist_ok=True)

model_path = artifacts / 'models' / 'spam_lstm_from_notebook.keras'
metrics_path = artifacts / 'reports' / 'metrics_from_notebook.txt'

model.save(model_path)
metrics_path.write_text(
    f'test_loss: {test_loss:.4f}\n'
    f'test_accuracy: {test_acc:.4f}\n',
    encoding='utf-8',
)

print(f'Model saved to: {model_path}')
print(f'Metrics saved to: {metrics_path}')